# **Processing Mass Spectrometry Proteomics Data**
---
## **Table of Contents**
- **Section (I) Mass Spectrometry in Proteomics: A Biophysical Overview**
    - **(I-1)** Top-Down vs. Bottom-Up Proteomics
    - **(I-2)** Core Components of a Mass Spectrometer
    - **(I-3)** From Protein to Spectrum: The Proteomics Pipeline
    - **(I-4)** Biophysical Principles at Work
    - **(I-5)** Why Mass Spectrometry Works for Proteomics
    - **(I-6)** Applications of Mass Spectrometry in Proteomics
- **Section (II) Examining Two "Shot-Gun" Proteomics Experiments**
    - **(II-1)**: Collecting Raw Mass Spec. Data
    - **(II-2)**: Metadata Extraction
    - **(II-3)**: Open Modification Search with Spectral Alignment and Grouping Engine (SAGE)
    - **(II-4)**: Constructing Your sage.conf File for Proteomics Data Analysis
    - **(II-5)**: Running SAGE for Peptide Quantification
    - **(II-6)**: Examine the Soybean peptide quantification file
    - **(II-7)**: Calculate and Examine the Protein Abundance Distributions for the Soybean Datasets
    - **(II-8)**: Differential Protein Expression from LFQ Mass Spectrometry Data of Soybean Dataset

## **Learning Objectives**
- Fundementals of mass spectrometry and how it is used in basic proteomics studies.
- Examine in depth how mass spectrometry proteomics data is analyzed, going from raw mass spectra to peptide and protein abundance estimations.
- Extract key technical meta data required for processing from the scientific publication.
- How to examine differential expression of proteins under different conditions.   

---
## **Calculate and Examine the Protein Abundance Distributions for the Datasets**
Now that you have obtain a list of peptides that have been confidently matched to proteins of interest we need to process further to obtain estimates of the protein level abundances. \

## (I) Imports and setup
Import some useful python packages for analyzing and plotting dataframes. 
Define the file path of the dataset you want and a label for the output files that are generated. 

In [10]:
import os, sys
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


## Define your root path as python notebooks require absolute paths to locate files
ROOT = os.path.abspath("../../")
print(f'Root path: {ROOT}')

## Choose which dataset you want to read in and comment out the other
DATA = "data/Human_SAGE_results/processed_sage_results.tsv" ## Human dataset
# DATA = "data/Soybean_SAGE_results/processed_sage_results.tsv" ## Soybean dataset
DATA = os.path.join(ROOT, DATA)
print(f'Using dataset: {DATA}')

## Check that the dataset file exists
if not os.path.isfile(DATA):
    print(f'Error: {DATA} not found.')
    raise FileNotFoundError
else:
    print(f'FILE EXISTS: {DATA}')

## Choose a label to differentiate any output files that are generated
dataLABEL = "Human" 
print(f'Using label: {dataLABEL}')


Root path: /Users/IanSitarik/git_repos/CURE2025_Comparative_massSpec
Using dataset: /Users/IanSitarik/git_repos/CURE2025_Comparative_massSpec/data/Human_SAGE_results/processed_sage_results.tsv
FILE EXISTS: /Users/IanSitarik/git_repos/CURE2025_Comparative_massSpec/data/Human_SAGE_results/processed_sage_results.tsv
Using label: Human


## (II) Filter the peptide tables
*Explanation*: Filters out low-confidence identifications. Only keeps peptides with q_value ≤ 0.05, controlling peptide-level false discovery. 
    
*Why*: Filtering by q-value ensures you are only using peptide measurements with statistical support propagating too many false positives would pollute protein-level inference.  
  
*Note*: The threshold (0.05) is conventional; students can try stricter (0.01) or looser as an exercise to see how discovery changes.  


In [13]:
## Load the data into a Pandas DataFrame and print the top 10 rows of the dataframe
df = pd.read_csv(DATA, sep="\t")
print(f'Loaded: {DATA}')
print(df.head())

## Filter the DataFrame based on q-value
df = df[df['q_value'] <= 0.05]  # Filter by q-value threshold
print(f'Number of confidently mapped peptides in {dataLABEL} samples: {len(df)}')

Loaded: /Users/IanSitarik/git_repos/CURE2025_Comparative_massSpec/data/Human_SAGE_results/processed_sage_results.tsv
                 peptide  charge  \
0               TAEENLDR       2   
1            YEEIDNAPEER       2   
2  AEDKPSTDLSAPVNGEATSQK       3   
3                 EALLGR       2   
4     TLPETLDPAEYNISPETR       2   

                                            proteins   q_value     score  \
0  sp|P43304|GPDM_HUMAN;tr|A0A140VJK2|A0A140VJK2_...  0.040656  0.687095   
1  sp|P49411|EFTU_HUMAN;tr|A0A384ME17|A0A384ME17_...  0.389773  0.474427   
2  sp|Q7Z4V5|HDGR2_HUMAN;tr|A0A087WX58|A0A087WX58...  0.284036  0.101991   
3  sp|P52565|GDIR1_HUMAN;tr|A0A0S2Z3D9|A0A0S2Z3D9...  0.041361  0.624547   
4  sp|O95168|NDUB4_HUMAN;tr|C9JXQ9|C9JXQ9_HUMAN;t...  0.316931  0.080647   

   spectral_angle  20141108_H3heavyN3light_1.mzML  \
0        0.884377                    3.971216e+06   
1        0.797473                    6.005948e+06   
2        0.505423                    0.000000e+00 

## (III) Select and clean relevant columns for analysis
*Explanation:*

* Constructs a subset of columns: retaining `peptide`, the original `proteins` annotation, `q_value`, and all other columns except technical metadata (`charge`, `score`, `spectral_angle`) which are not needed for quantification.
* The `proteins` field is in UniProt format like `sp|P12345|PROT_NAME;tr|P12345|PROT_NAME;...`.
* For each peptide make a row copy for every protein it could possibly map to.
* Drops any peptides that failed to parse into a valid UniProt accession.
  
*Why:*
Reduces clutter and focuses downstream computation on actual intensity data and needed identifiers.  

*Teaching note:* Students can be asked what happens if peptides map to multiple proteins (this script doesn’t resolve shared peptides explicitly—it aggregates all evidence under each protein) and discuss how “razor” or parsimony approaches differ.

In [14]:
## Select and clean Human columns
import re
columns_of_interest = ['peptide', 'proteins', 'q_value'] + [col for col in df.columns if col not in ['peptide', 'proteins', 'q_value', 'charge', 'score', 'spectral_angle']]
clean_df = df[columns_of_interest]

## Function to take each peptide and make duplicate rows for every protein it maps to with confidence
def expand_by_uniprot(df):
    """
    Expands the DataFrame such that each row is duplicated for each UniProt ID
    found in the 'proteins' column, and a new column 'uniprot' is added with that ID.
    """
    rows = []
    
    for _, row in df.iterrows():
        # Split the proteins string by ';' and extract UniProt IDs using regex
        entries = row['proteins'].split(';')
        for entry in entries:
            match = re.search(r'\|([A-Z0-9]+)\|', entry)
            if match:
                uniprot_id = match.group(1)
                new_row = row.copy()
                new_row['uniprot'] = uniprot_id
                rows.append(new_row)
    
    # Combine all expanded rows into a new DataFrame
    expanded_df = pd.DataFrame(rows)
    return expanded_df

## Expand the DataFrame so that each protein it is matched to
clean_expanded_df = expand_by_uniprot(clean_df)

## drop the columns we dont need
clean_expanded_df.drop(columns=['proteins'], inplace=True)

## Drop peptides without a recognized protein mapping
clean_expanded_df.dropna(subset=['uniprot'], inplace=True)
clean_expanded_df = clean_expanded_df.reset_index(drop=True)

## Display the first few rows of the cleaned and expanded DataFrame
print(f'First few rows of the cleaned and expanded DataFrame:')
print(clean_expanded_df.head())

First few rows of the cleaned and expanded DataFrame:
    peptide   q_value  20141108_H3heavyN3light_1.mzML  \
0  TAEENLDR  0.040656                    3.971216e+06   
1  TAEENLDR  0.040656                    3.971216e+06   
2  TAEENLDR  0.040656                    3.971216e+06   
3  TAEENLDR  0.040656                    3.971216e+06   
4  TAEENLDR  0.040656                    3.971216e+06   

   20141108_H2heavyN2light_1.mzML  20141108_H1heavyN1light_1.mzML  \
0                    56459.075712                     3130.584116   
1                    56459.075712                     3130.584116   
2                    56459.075712                     3130.584116   
3                    56459.075712                     3130.584116   
4                    56459.075712                     3130.584116   

   20141108_H1heavyN1light_2.mzML  20141108_H2heavyN2light_2.mzML  \
0                   904991.759158                    4.358405e+06   
1                   904991.759158                 

## (IV) Estimate Protein Abundances
*Explanation:*

* Groups all peptides assigned to each UniProt accession and collapses their intensities per sample by taking the **median** across peptides.

*Why median?*
Median is robust to outliers (e.g., aberrant peptide measurements) and is a simple estimate of protein abundance from multiple peptide surrogates.

*Note:* Alternatives include mean, weighted mean (by peptide quality), or more sophisticated models (like MaxLFQ’s ratio-based inference). Students can compare results of median vs mean as an exploration.
https://www.nature.com/articles/nmeth.3901.pdf

In [15]:
## Identify the peptide intensity columns associated with the treatment and control groups
# 20141108_H3heavyN3light_1.mzML	20141108_H2heavyN2light_1.mzML	20141108_H1heavyN1light_1.mzML	20141108_H1heavyN1light_2.mzML	20141108_H2heavyN2light_2.mzML	20141108_H3heavyN3light_2.mzML	20141108_H1heavyN1light_3.mzML	20141108_H2heavyN2light_3.mzML	20141108_H3heavyN3light_3.mzML	20141108_H1heavyN1light_4.mzML	20141108_H2heavyN2light_4.mzML	20141108_H3heavyN3light_4.mzML	20141108_H2heavyN2light_5.mzML	20141108_H1heavyN1light_5.mzML

groups = {'CONTROL': ['Soybean_LPBiomaker_HP1.mzML', 'Soybean_LPBiomaker_HP2.mzML', 'Soybean_LPBiomaker_HP3.mzML'], 
          'TREATMENT': ['Soybean_LPBiomaker_LP1.mzML', 'Soybean_LPBiomaker_LP2.mzML', 'Soybean_LPBiomaker_LP3.mzML']}


# Collapse to protein-level abundance for each replicate
protein_sample_abundance = (Soybean_clean_expanded_lfq.groupby('uniprot')[lp_cols + hp_cols].median())
    
print('Protein Abundance Dataframe')
display_df_scroll(protein_sample_abundance, max_height="400px")

NameError: name 'Soybean_clean_expanded_lfq' is not defined

**Explanation:**

* Groups all peptides assigned to each UniProt accession and collapses their intensities per sample by taking the **median** across peptides.

**Why median?**
Median is robust to outliers (e.g., aberrant peptide measurements) and is a simple estimate of protein abundance from multiple peptide surrogates.

**Teaching note:** Alternatives include mean, weighted mean (by peptide quality), or more sophisticated models (like MaxLFQ’s ratio-based inference). Students can compare results of median vs mean as an exploration.
https://www.nature.com/articles/nmeth.3901.pdf

## 5. Compute Stats of Sample distributions

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
import textwrap

## Function to compute the stats for each sample column 
def compute_stats(series, label):
    """Return a dict and dataframe of summary statistics for a 1D numeric pandas Series."""
    clean = series.dropna()
    if clean.empty:
        return {
            "count": 0,
            "mean": np.nan,
            "median": np.nan,
            "mode": [],
            "variance": np.nan,
            "quantiles": {}
        }
    # mode: use pandas.Series.mode() for compatibility
    mode_vals = clean.mode()

    results = {
        "count": int(clean.size),
        "mean": float(clean.mean()),
        "median": float(clean.median()),
        "mode": [float(m) for m in mode_vals] if not mode_vals.empty else [],
        "variance": float(clean.var(ddof=1)),  # sample variance
        "quantiles": {
            "0.0": float(clean.quantile(0.0)),
            "0.25": float(clean.quantile(0.25)),
            "0.5": float(clean.quantile(0.5)),
            "0.75": float(clean.quantile(0.75)),
            "1.0": float(clean.quantile(1.0)),
        }
    }
    results_df = {"count": int(clean.size),
                  "mean": float(clean.mean()),
                  "median": float(clean.median()),
                  "mode": [float(m) for m in mode_vals] if not mode_vals.empty else [],
                  "variance": float(clean.var(ddof=1)),  # sample variance,
                  "Q0.0": float(clean.quantile(0.0)),
                  "Q0.25": float(clean.quantile(0.25)),
                  "Q0.5": float(clean.quantile(0.5)),
                  "Q0.75": float(clean.quantile(0.75)),
                  "Q1.0": float(clean.quantile(1.0))}
    results_df = pd.DataFrame(results_df)
    results_df['file'] = label
    
    return results_df, results
    
# Compute per-sample stats
stats_dict = {}
stats_dfs = []
for c in lp_cols + hp_cols:
    stats_df, stats_dict[c] = compute_stats(protein_sample_abundance[c], c)
    stats_dfs.append(stats_df)
                
stats_df = pd.concat(stats_dfs)
display_df_scroll(stats_df, max_height="400px")

## 6. Make Box Plots of each Sample Separately

In [ ]:
def plot_sample_boxplots(df, sample_cols, label_map, output_path=None, log_scale=False):
    """
    Box plots for individual samples with custom labels.

    Parameters
    ----------
    df : pd.DataFrame
        Protein abundance table (rows=proteins, cols=samples).
    sample_cols : List[str]
        Columns in df to plot.
    label_map : Dict[str,str]
        Mapping from sample column name to custom label for x-axis.
    output_path : str or None
        If given, saves figure to this path.
    log_scale : bool
        If True, plot log2(abundance + small pseudocount).
    """
    # Prepare data array for boxplot
    data = []
    labels = []
    for col in sample_cols:
        series = df[col]
        if log_scale:
            series = np.log2(series.replace(0, np.nan) + 1e-6)
        data.append(series.dropna().values)
        labels.append(label_map.get(col, col))

    fig, ax = plt.subplots(figsize=(1.5 * len(sample_cols) + 2, 6))
    bp = ax.boxplot(data, patch_artist=True, widths=0.6, manage_ticks=False)

    # Light styling: gray boxes with white medians
    for box in bp['boxes']:
        box.set(facecolor='#c0c0c0', alpha=0.7, edgecolor='black')
    for whisker in bp['whiskers']:
        whisker.set(color='black', linewidth=1)
    for cap in bp['caps']:
        cap.set(color='black', linewidth=1)
    for median in bp['medians']:
        median.set(color='white', linewidth=2)
    for flier in bp['fliers']:
        flier.set(marker='o', alpha=0.4, markersize=4)

    ax.set_xticks(np.arange(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ylabel = "log2(abundance)" if log_scale else "abundance"
    ax.set_ylabel(ylabel)
    ax.set_title("Box plots of individual samples")
    ax.grid(axis='y', linestyle=':', linewidth=0.5, alpha=0.7)
    ax.set_ylim(-10000000, 4e8)
    fig.tight_layout()
    plt.show()
    # if output_path:
    #     fig.savefig(output_path, dpi=300)
    #     print(f"Saved violin plot to {output_path}")
    return fig, stats_dict

# Assume protein_abundance_df is your dataframe with uniprot index
sample_cols = [
    "Soybean_LPBiomaker_LP1.mzML",
    "Soybean_LPBiomaker_LP2.mzML",
    "Soybean_LPBiomaker_LP3.mzML",
    "Soybean_LPBiomaker_HP1.mzML",
    "Soybean_LPBiomaker_HP2.mzML",
    "Soybean_LPBiomaker_HP3.mzML",
]
label_map = {
    "Soybean_LPBiomaker_LP1.mzML": "LP1",
    "Soybean_LPBiomaker_LP2.mzML": "LP2",
    "Soybean_LPBiomaker_LP3.mzML": "LP3",
    "Soybean_LPBiomaker_HP1.mzML": "HP1",
    "Soybean_LPBiomaker_HP2.mzML": "HP2",
    "Soybean_LPBiomaker_HP3.mzML": "HP3",
}

fig, stats = plot_sample_boxplots(protein_sample_abundance, sample_cols, label_map,
                                output_path="sample_boxplot.png", log_scale=False)

- Do you observe any noticable differences in the distribution of protein abundances between technical replicates under the same condition (i.e. LP and HP)?
- How about between conditions?
- Do these plots make sense with the statistics you calculated previously?
- What does this tell you about the proteomes of these two organisms and does this make sense for two very different organisms?

## **(II-8): Differential Protein Expression from LFQ Mass Spectrometry Data of Soybean Dataset**

*Using a robust, classical pipeline with fold-change inference via per-protein statistics*


**Explanation:**

* `welch_test`: performs Welch’s t-test comparing the LP and HP sample groups for a given protein (row of the protein-level matrix). `equal_var=False` allows unequal variances—a safer default in proteomics. `nan_policy='omit'` means missing values don’t crash the test.

**Teaching note:** Welch’s t-test is generally powerful when sample sizes are modest and distributions are approximately symmetric; the Mann–Whitney test is robust to non-normality but tests for difference in distribution location in a different way. It’s good practice to examine data (e.g., via violin/box plots) to decide if assumptions are reasonable.

## 1. Protein-level abundance statistics 

In [ ]:
# calculate the protein level abundance statistics
protein_abund_stats = {}
for group_name, cols in groups.items():
    sub = protein_sample_abundance[cols]

    # Compute per-row stats; skip NaNs naturally
    protein_abund_stats[f"{group_name}_mean"] = sub.mean(axis=1, skipna=True)
    protein_abund_stats[f"{group_name}_std"] = sub.std(axis=1, ddof=1, skipna=True)
    protein_abund_stats[f"{group_name}_var"] = sub.var(axis=1, ddof=1, skipna=True)
    protein_abund_stats[f"{group_name}_median"] = sub.median(axis=1, skipna=True)
    
protein_abund_stats = pd.DataFrame(protein_abund_stats)
protein_abund_stats['Fold2AbundChange'] = np.log2(protein_abund_stats[f"LP_mean"]/protein_abund_stats[f"HP_mean"])
# print(protein_abund_stats)

# add fold2 change in abundance means to protein abundance df
protein_sample_abundance['Fold2AbundChange'] = np.log2(protein_abund_stats[f"LP_mean"]/protein_abund_stats[f"HP_mean"])
display_df_scroll(protein_sample_abundance, max_height="400px")

## 2. Standard Scale Renormalization of Protein Abundance

In [ ]:
# Renormalize the columns 
LP_mean = protein_sample_abundance[lp_cols].values.flatten().mean()
HP_mean = protein_sample_abundance[hp_cols].values.flatten().mean()
LP_std = protein_sample_abundance[lp_cols].values.flatten().std()
HP_std = protein_sample_abundance[hp_cols].values.flatten().std()

renorm_protein_sample_abundance = protein_sample_abundance.copy()
renorm_protein_sample_abundance[lp_cols] = (protein_sample_abundance[lp_cols] - LP_mean) / LP_std
renorm_protein_sample_abundance[hp_cols] = (protein_sample_abundance[hp_cols] - HP_mean) / HP_std

# calculate the renormalized protein level abundance statistics
renorm_protein_abund_stats = {}
for group_name, cols in groups.items():
    sub = renorm_protein_sample_abundance[cols]

    # Compute per-row stats; skip NaNs naturally
    renorm_protein_abund_stats[f"{group_name}_mean"] = sub.mean(axis=1, skipna=True)
    renorm_protein_abund_stats[f"{group_name}_std"] = sub.std(axis=1, ddof=1, skipna=True)
    renorm_protein_abund_stats[f"{group_name}_var"] = sub.var(axis=1, ddof=1, skipna=True)
    renorm_protein_abund_stats[f"{group_name}_median"] = sub.median(axis=1, skipna=True)
    
renorm_protein_abund_stats = pd.DataFrame(renorm_protein_abund_stats)
renorm_protein_abund_stats['RenomAbundChange'] = (renorm_protein_abund_stats[f"LP_mean"] - renorm_protein_abund_stats[f"HP_mean"]) 

## Add the fold change columns to the renormalized protein abundance df
renorm_protein_sample_abundance['RenomAbundChange'] = (renorm_protein_abund_stats[f"LP_mean"] - renorm_protein_abund_stats[f"HP_mean"])
# renorm_protein_sample_abundance['Fold2AbundChange'] = np.log2(protein_abund_stats[f"LP_mean"]/protein_abund_stats[f"HP_mean"])
display_df_scroll(renorm_protein_sample_abundance, max_height="400px")

**Explanation:**

* Computes global mean and standard deviation separately for LP and HP samples (flattening all proteins × replicates).
* Z-scores each group independently: subtract mean, divide by standard deviation.

**Why:**
This rescales each condition so that its overall distribution is standardized. It removes global scale differences that might come from batch effects or loading differences.

**Caveat:**
Standardizing separately means the two groups are no longer directly on the same absolute scale—this could obscure true global shifts. Normally, for differential testing you'd either:

* Normalize all samples together (e.g., median normalization across all), or
* Use ratio-based methods where differences are preserved without groupwise rescaling.

**Teaching exercise:** Have students try skipping this step or doing a joint normalization and compare which proteins are called significant.

## 3. Statistical testing (Welch’s t-test)

In [ ]:
from scipy import stats


def welch_per_row_inplace(df, group_a_cols, group_b_cols,
                          tstat_col="t_stat", pvalue_col="p_value", df_col="df_welch"):
    """
    Perform Welch's t-test on each row between two column groups and
    return a dataframe with added columns for t-statistic, p-value, and Welch df.

    Parameters
    ----------
    df : pd.DataFrame
        Rows are entities (e.g., proteins), columns are samples with numeric values.
    group_a_cols : list of str
        Column names in df belonging to group A.
    group_b_cols : list of str
        Column names in df belonging to group B.
    tstat_col, pvalue_col, df_col : str
        Names of the added columns for output.

    Returns
    -------
    df_out : pd.DataFrame
        A new DataFrame with the same index and original columns plus the three result columns.
    """
    # Validate columns exist
    for c in group_a_cols + group_b_cols:
        if c not in df.columns:
            raise KeyError(f"Column {c!r} not found in dataframe")

    # Prepare containers
    tstats = []
    pvalues = []
    dfs = []

    for idx, row in df.iterrows():
        a = row[group_a_cols].dropna().astype(float)
        b = row[group_b_cols].dropna().astype(float)
        n_a = a.size
        n_b = b.size
        var_a = a.var(ddof=1) if n_a > 1 else np.nan
        var_b = b.var(ddof=1) if n_b > 1 else np.nan

        # Default outputs
        t_stat = np.nan
        p_value = np.nan
        df_welch = np.nan

        if n_a >= 1 and n_b >= 1:
            # Welch t-test (scipy already uses Welch–Satterthwaite internally)
            res = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
            t_stat = res.statistic
            p_value = res.pvalue

            # Compute Welch–Satterthwaite df manually if possible
            if n_a > 1 and n_b > 1 and not np.isnan(var_a) and not np.isnan(var_b):
                s1 = var_a
                s2 = var_b
                num = (s1 / n_a + s2 / n_b) ** 2
                denom = ( (s1 ** 2) / (n_a ** 2 * (n_a - 1)) ) + ( (s2 ** 2) / (n_b ** 2 * (n_b - 1)) )
                df_welch = num / denom if denom > 0 else np.nan

        tstats.append(t_stat)
        pvalues.append(p_value)
        dfs.append(df_welch)

    df_out = df.copy()
    df_out[tstat_col] = tstats
    df_out[pvalue_col] = pvalues
    df_out[df_col] = dfs

    return df_out

renorm_protein_sample_abundance = welch_per_row_inplace(renorm_protein_sample_abundance, lp_cols, hp_cols)

display_df_scroll(renorm_protein_sample_abundance, max_height="400px")

**Explanation:**

* Applies Welch’s t-test per protein, comparing the standardized LP vs HP vectors (each of length 3).
* Stores the test statistic and raw p-value.

**Why Welch’s test?**
It does not assume equal variance between groups—a prudent choice for biological data where heteroskedasticity is common.

**Teaching note:** Point out that with only 3 replicates per group, degrees of freedom are low; encourage students to visualize variance and consider the effect of small sample size on power and false negatives.

## 4. Ranking and multiple testing correction

In [ ]:
# sort by p-value smallest to largest
renorm_protein_sample_abundance.sort_values(by='p_value', inplace=True)
renorm_protein_sample_abundance.reset_index(inplace=True)

# Adjust p-values for multiple testing
from statsmodels.stats.multitest import multipletests
renorm_protein_sample_abundance['adj_pval'] = multipletests(
    renorm_protein_sample_abundance['p_value'], method='fdr_bh')[1]
display_df_scroll(renorm_protein_sample_abundance, max_height="400px")

**Explanation:**

* Sorts proteins by raw significance (smallest p-value first) to prioritize candidates.
* Applies Benjamini–Hochberg FDR correction (`fdr_bh`) to control the expected proportion of false discoveries among calls.

**Why:**
Thousands of proteins may be tested; without correction, the number of false positives at p<0.05 would be unacceptably high. FDR balances discovery with error control.

**Teaching note:** Discuss differences among correction methods (`bonferroni`, `fdr_bh`, `fdr_by`, etc.) and when each is appropriate.

## 5. Extract significant proteins

In [ ]:
# select the significant proteins
significant_proteins = renorm_protein_sample_abundance[renorm_protein_sample_abundance['adj_pval'] < 0.05]

display_df_scroll(significant_proteins, max_height="400px")

**Explanation:**

* Filters proteins whose adjusted p-value is below 0.05, declaring them differentially abundant between LP and HP.
* Prints their UniProt accession IDs.

**Why:**
This yields a candidate list for biological interpretation: proteins whose standardized abundance differs beyond expected noise.

## 6. Volcano plots

A **volcano plot** is a scatter plot that simultaneously displays both **magnitude of change** and **statistical significance** for each protein (or peptide) when comparing two conditions. It’s a rapid visual filter to highlight candidates that are both substantially and confidently different.

### Axes:

* **X-axis:** Fold change (here, `Fold2AbundChange`). This is usually on a log2 scale (positive means higher in condition B, negative means higher in condition A). It reflects **effect size** — how much a protein’s abundance changes.
* **Y-axis:** $-\log_{10}(\text{adjusted p-value})$ from a statistical test (e.g., Welch’s t-test on replicate abundances). Higher values mean stronger statistical evidence against the null (i.e., more significant). Using the negative log makes small p-values (strong significance) appear at the top.

### What Each Point Represents:

Each point is a protein. Its horizontal position is how big the abundance change is, and its vertical position is how unlikely that change is due to random noise (after correcting for multiple comparisons).

### Coloring & Thresholds (as in your plot)

* **Horizontal line at adjusted p-value = 0.05:**
  Marks the significance cutoff. Points above this line are statistically significant (adj p-value < 0.05).

* **Vertical line at 0 fold change:**
  Separates proteins that increase (right) from those that decrease (left) between the two conditions.

* **Color scheme:**

  * **Bright red:** Significant *and* large change (|Fold2AbundChange| ≥ 1). These are high-confidence, high-effect candidates.
  * **Pink:** Significant but small change (|Fold2AbundChange| < 1). Statistically reliable but modest in magnitude—could be biologically meaningful if consistent across pathways.
  * **Black:** Not significant (adj p-value ≥ 0.05), regardless of fold change—these could be noise or underpowered.


In [ ]:

signif_cutoff=0.05
fc_threshold=1.0
alpha=0.7
fc_col = 'Fold2AbundChange'
pval_col = 'adj_pval'

# Prepare axes values
x = renorm_protein_sample_abundance[fc_col].astype(float)
y = -np.log10(renorm_protein_sample_abundance[pval_col].astype(float))

sig = renorm_protein_sample_abundance[pval_col] < signif_cutoff
large_fc = x.abs() >= fc_threshold
small_fc = (~large_fc) & sig

# Base scatter: non-significant
fig, ax = plt.subplots(figsize=(7,6))
ax.scatter(x[~sig], y[~sig], c="black", s=30, alpha=alpha, label="not sig")

# significant but small FC
ax.scatter(x[small_fc], y[small_fc], c="lightpink", edgecolor="none", s=40, alpha=alpha, label=f"sig, |FC| < {fc_threshold}")

# significant large FC
ax.scatter(x[sig & large_fc], y[sig & large_fc], c="red", edgecolor="none", s=50, alpha=alpha, label=f"sig, |FC| ≥ {fc_threshold}")

# Threshold lines
ax.axhline(-np.log10(signif_cutoff), linestyle="--", color="gray", linewidth=1, label=f"p_adj = {signif_cutoff}")
ax.axvline(0, linestyle="--", color="gray", linewidth=1)

ax.set_xlabel("Fold2AbundChange")
ax.set_ylabel(r"$-\log_{10}(\mathrm{adj\_pval})$")
ax.set_title("Differential Protein Abundance Volcano Plot")
ax.legend(frameon=False, fontsize=8)
ax.grid(True, linestyle=":", linewidth=0.5, alpha=0.6)
plt.show()


[1]: https://github.com/lazear/sage?utm_source=chatgpt.com "lazear/sage: Proteomics search & quantification so fast that ... - GitHub"
[2]: https://lazear.github.io/sage/?utm_source=chatgpt.com "Proteomics searching so fast it seems like Magic"
[3]: https://www.researchgate.net/publication/374639418_Sage_An_Open-Source_Tool_for_Fast_Proteomics_Searching_and_Quantification_at_Scale?utm_source=chatgpt.com "Sage: An Open-Source Tool for Fast Proteomics Searching and ..."
[4]: https://pubmed.ncbi.nlm.nih.gov/37819886/?utm_source=chatgpt.com "Sage: An Open-Source Tool for Fast Proteomics ... - PubMed"
[5]: https://www.sciencedirect.com/science/article/pii/S1535947624000884?utm_source=chatgpt.com "Rescoring Peptide Spectrum Matches: Boosting Proteomics ..."
[6]: https://pmc.ncbi.nlm.nih.gov/articles/PMC4125737/?utm_source=chatgpt.com "Conserved Peptide Fragmentation as a Benchmarking Tool for Mass ..."